# Review Translation: Brazilian E-Commerce Dataset

Background

The Olist Brazilian e-commerce dataset contains customer reviews written primarily in Portuguese. Since the subsequent review analysis is intended to be conducted in English, the original Portuguese review comments need to be translated before they are used in downstream natural language processing tasks.

### Methodology


The translation workflow consists of the following procedures:

1. Load and prepare the review data.

2. Load the translation model and tokenizer.

3. Identify previously translated reviews.

4. Translate reviews in batches.
Reviews are tokenized and translated batch by batch. The generated token sequences are then decoded into English text. The translation output is checked to ensure that each input review has a corresponding translation.
Save translations incrementally.
Each completed batch is immediately appended to a translation CSV file rather than keeping the entire translation result in memory. Processing progress and batch execution time are also recorded.

5. Map translations back to the original review dataset.
Once the translation process is complete, the saved translations are loaded and mapped back to the original review records using the Portuguese review text as the matching key.

6. Check and handle missing or non-informative reviews.
The translated dataset is checked for review comments that do not have a corresponding English translation. Values such as "N/a", "N/D", and "Nenhuma" are treated as missing rather than meaningful review content. Any remaining untranslated review comments are processed through a final fallback translation step.

7. Save the final translated dataset.
The completed dataset is exported with the original order_id, the original Portuguese review comment, and the corresponding English translation so that it can be used in subsequent analysis.

### 1. Load and prepare the review data.

In [2]:
import pandas as pd
import os
import time

reviews = pd.read_csv(r'..\data\raw\olist_order_reviews_dataset.csv', index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")

Comments with NA values are excluded from translation. Duplicate review texts are also removed so that each unique Portuguese comment is translated only once. The resulting translations can subsequently be mapped back to all corresponding review records, substantially reducing redundant computation.

In [ ]:
unique_texts = (
    reviews["review_comment_message"]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Unique reviews: {len(unique_texts):,}")

Unique reviews: 36,159


### 2. Load the translation model and tokenizer.

A pretrained Portuguese-to-English neural machine translation model is loaded using an INT8-quantized ONNX implementation. The corresponding tokenizer is used to convert Portuguese review text into the numerical representation required by the model.

A dedicated neural machine translation model is sufficient for this task because the objective is to translate Portuguese review text rather than perform general-purpose language generation. The INT8 ONNX implementation reduces the computational and memory requirements of local inference, making it more practical for translating the dataset on a local machine.

In [ ]:
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM

model_id = "R4kSo1997/opus-mt-pt-en-onnx-int8"

tokenizer = AutoTokenizer.from_pretrained(
    "Helsinki-NLP/opus-mt-roa-en"
)

model = ORTModelForSeq2SeqLM.from_pretrained(
    model_id,
    encoder_file_name="encoder_model_quantized.onnx",
    decoder_file_name="decoder_model_quantized.onnx",
    decoder_with_past_file_name="decoder_with_past_model_quantized.onnx",
)

c:\Users\19148\anaconda3\envs\pydata-book\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


### 3. Identify previously translated reviews.

The complete translation task can require many hours of continuous computation. During such a long-running process, the notebook or computing environment may be interrupted before all reviews have been translated. Saving and reusing intermediate results makes the pipeline restartable and fault-tolerant, while also substantially reducing unnecessary computation after an interruption.

Check whether a previously generated translation file exists before starting translation.

In [ ]:
translation_file = r"../data/processed/review_translations.csv"

if os.path.exists(translation_file):

    saved = pd.read_csv(translation_file)

    translated_texts = set(
        saved["review_comment_message"]
    )

    print(
        f"Found {len(translated_texts):,} "
        f"existing translations."
    )

else:

    translated_texts = set()

    print("No existing translation file.")

Found 36,159 existing translations.
Remaining reviews: 0
shorter : 0
short   : 0
medium  : 0
long    : 0

Starting shorter group (0 reviews, batch size = 5)

Starting short group (0 reviews, batch size = 5)

Starting medium group (0 reviews, batch size = 5)

Starting long group (0 reviews, batch size = 3)


Reviews that have already been translated are identified and excluded from the remaining workload.

In [ ]:
remaining = unique_texts[
    ~unique_texts.isin(translated_texts)
].reset_index(drop=True)

print(
    f"Remaining reviews: "
    f"{len(remaining):,}"
)


### 4. Batch translate based on length.


**Group reviews based on length**

Reviews are grouped according to their character length before batching. Because character length is generally correlated with the number of tokens produced by the tokenizer, this grouping provides a practical approximation for grouping reviews with similar token lengths. Reviews with similar lengths are processed together so that padding required by `padding=True` is reduced.

Consider two possible batches:

Batch A: 20, 21, 23, 25 tokens  
Batch B: 20, 25, 80, 120 tokens

With `padding=True`, Batch A is padded only slightly, because all reviews have similar lengths. In contrast, the shorter reviews in Batch B must be padded substantially to match the 120-token review. A large portion of the computation performed for those shorter reviews therefore corresponds to padding rather than actual text. Thus, grouping reviews of similar length helps reduce this padding overhead.

Padding is necessary because the model requires inputs in a uniform tensor shape in order to perform batch processing. Without it, each review would have a different sequence length, preventing them from being stacked into a single batch tensor. As a result, the model would either fail to process multiple reviews together or would be forced to process them one-by-one, eliminating the efficiency benefits of batching. Padding is therefore necessary to enable parallel processing, even though it introduces some additional computation for padded tokens.

In [ ]:
lengths = remaining.str.len()

groups = {
    "shorter": remaining[lengths < 30],
    "short": remaining[
        (lengths >= 30) & (lengths < 60)
    ],
    "medium": remaining[
        (lengths >= 60) & (lengths < 100)
    ],
    "long": remaining[lengths >= 100]
}


for name, group in groups.items():
    print(f"{name:8s}: {len(group):,}")

Batch sizes are configured empirically according to available computational resources. Smaller batches are used for longer reviews because they require more computation and memory during generation.

In [ ]:
batch_sizes = {
    "shorter": 5,
    "short": 5,
    "medium": 5,
    "long": 3
}

In [ ]:
total_remaining = len(remaining)
processed_total = 0

**Translation flow:**

- Tokenization & Translation: `tokenizer(...)` converts the review strings into token IDs and attention masks, which are then passed to `model.generate()`.

- Decoding: The generated token sequences are decoded back into English text. Special tokens used by the sequence-to-sequence model are removed with `skip_special_tokens=True`, producing clean text suitable for storage as review comments.

- Translation Integrity Check: After each batch is translated, the notebook checks that the number of generated translations matches the number of input reviews.

- Incremental Storage: Each completed batch is converted into a temporary DataFrame containing the original review and its translation, then immediately appended to the CSV file. Because completed batches are written immediately to disk, an interrupted run does not require previously completed translations to be recomputed. When the notebook is restarted, the existing CSV is read and the corresponding Portuguese review texts are excluded from the remaining workload.  
`mode="a"` appends rather than overwriting the existing CSV. The header is written only when the CSV is first created.

- Release current batch: Only the current batch is kept in memory during translation. After the batch has been written to disk, the temporary objects associated with that batch are deleted, allowing their memory to be released before processing the next batch. 

In [ ]:
for group_name, group in groups.items():

    batch_size = batch_sizes[group_name]

    print(
        f"\nStarting {group_name} group "
        f"({len(group):,} reviews, "
        f"batch size = {batch_size})"
    )

    for i in range(
        0,
        len(group),
        batch_size
    ):

        # This automatically handles the final
        # incomplete batch.
        batch = group.iloc[
            i:i + batch_size
        ].tolist()

        batch_start = time.time()

        # Tokenization
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        # Translation
        outputs = model.generate(
            **inputs
        )

        translations = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        # Safety check
        if len(batch) != len(translations):
            raise RuntimeError(
                f"Expected {len(batch)} translations, "
                f"but received {len(translations)}."
            )

        # Create current batch only
        batch_df = pd.DataFrame({
            "review_comment_message": batch,
            "review_comment_en": translations
        })

        # Append directly to CSV
        batch_df.to_csv(
            translation_file,
            mode="a",
            header=not os.path.exists(
                translation_file
            ),
            index=False
        )

        elapsed = time.time() - batch_start

        processed_total += len(batch)

        print(
            f"{processed_total:,}/"
            f"{total_remaining:,} "
            f"({processed_total / total_remaining:.1%}) "
            f"| {group_name} "
            f"| {len(batch)} reviews "
            f"| {elapsed:.1f}s"
        )

        # Release current batch
        del inputs
        del outputs
        del translations
        del batch_df
        del batch

### 5. Map translations back to the original review dataset.

Once the translation process is complete, the saved translations are loaded and mapped back to the original review records using the Portuguese review text as the matching key.


Translation is performed at the unique-text level for efficiency, but the downstream analysis is performed at the review-record level. Mapping the translations back to the original dataset restores the original structure while retaining the computational benefits of translating unique texts only once.

In [6]:
translations = pd.read_csv(translation_file)
translations.head()

,review_comment_message,review_comment_en
0,Recebi bem antes do prazo estipulado.,I got it well before the deadline.
1,Parabéns lojas lannister adorei comprar pela I...,Congratulations Lannister shops I love to buy ...
2,aparelho eficiente. no site a marca do aparelh...,efficient device. on the site the brand of the...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n","But a little, stopping... for the value of you..."
4,"Vendedor confiável, produto ok e entrega antes...","Reliable seller, product ok and delivery befor..."


In [7]:
translation_map = dict(
    zip(
        translations["review_comment_message"],
        translations["review_comment_en"]
    )
)

reviews["review_comment_en"] = (
    reviews["review_comment_message"]
    .map(translation_map)
)

### 6. Check and handle missing or non-informative reviews.

The translated dataset is checked for review comments that do not have a corresponding English translation.

In [8]:
missing = reviews[
    reviews["review_comment_message"].notna()
    & reviews["review_comment_en"].isna()
]

print(f"Missing translations: {len(missing):,}")

Missing translations: 3


In [9]:
missing[[
    "review_comment_message",
    "review_comment_en"
]]

,review_comment_message,review_comment_en
order_id,,
195aef69980a745473d3fa7be3523f58,N/a,NaN
b4700fc91bb3fbc155bfc04ae821ba04,N/D,NaN
2e7ca2344bf0db90bba866fa90409cfd,Nenhuma,NaN


Values such as "N/a", "N/D", and "Nenhuma" are treated as missing rather than meaningful review content. 

In [10]:
reviews.loc[
    reviews["review_comment_message"].isin(
        ["N/a", "N/D", "Nenhuma"]
    ),
    "review_comment_message"
] = pd.NA

Any remaining untranslated review comments are processed through a final fallback translation step.

In [ ]:
missing = reviews[
    reviews["review_comment_message"].notna()
    & reviews["review_comment_en"].isna()
]

if len(missing) > 0:
    inputs = tokenizer(
        missing['review_comment_message'].tolist(),
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Translation
    outputs = model.generate(
        **inputs
    )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    reviews.loc[
        reviews["review_comment_message"].isin(
            missing['review_comment_message'].tolist()
        ),
        "review_comment_en"
    ] = translations

### 7. Save the final translated dataset.

The completed dataset is exported with the original order_id, the original Portuguese review comment, and the corresponding English translation so that it can be used in subsequent analysis..

In [14]:
reviews.to_csv(
    r"../src/olist_reviews_translated.csv",
    index=True,
    index_label="order_id"
)

### Summary and Next Steps

**Summary**

The translation workflow converts the Portuguese review comments in the Olist dataset into English through a local neural machine translation pipeline. To reduce redundant computation, translation is performed only once for each unique Portuguese review text. The resulting translations are then mapped back to all review records containing the same text, preserving the original record-level structure. Previously translated comments are skipped, untranslated reviews are grouped by length, and the remaining comments are processed in controlled batches using an INT8 ONNX model.

Translations are saved incrementally to provide both memory efficiency and a restartable workflow. After translation, the results are mapped back to the original review dataset, non-informative comments are treated as missing, and any remaining untranslated reviews are handled through a fallback translation step.

The final dataset preserves the original review information while adding an English translation column, providing the basis for subsequent English-language review analysis such as aspect extraction and sentiment analysis.

**Next Steps**
The translated reviews will be used in the next stage, assisting in constructing a review-based rating system. The analysis will first identify and define candidate aspects from the reviews using sentence embeddings and nearest-neighbor inspection. These candidate aspects will then be used in an LLM-based extraction step to identify aspect-level evaluations related to the Product, Seller, and Delivery dimensions.

The resulting aspect-level evaluations will subsequently be aggregated to construct ratings for these three dimensions.
